In [1]:
import pyautogui as pyg
pyg.sleep(2)
basex=542
basey=425
gap=40

In [2]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)

0 0 542 425 RGB(red=88, green=86, blue=86)
0 1 582 425 RGB(red=57, green=54, blue=66)
0 2 622 425 RGB(red=53, green=52, blue=64)
0 3 662 425 RGB(red=57, green=56, blue=71)
0 4 702 425 RGB(red=67, green=67, blue=81)
0 5 742 425 RGB(red=31, green=31, blue=31)
0 6 782 425 RGB(red=31, green=31, blue=31)
0 7 822 425 RGB(red=31, green=31, blue=31)
1 0 542 465 RGB(red=230, green=230, blue=230)
1 1 582 465 RGB(red=108, green=108, blue=108)
1 2 622 465 RGB(red=31, green=31, blue=31)
1 3 662 465 RGB(red=242, green=244, blue=247)
1 4 702 465 RGB(red=242, green=244, blue=247)
1 5 742 465 RGB(red=242, green=244, blue=247)
1 6 782 465 RGB(red=255, green=255, blue=255)
1 7 822 465 RGB(red=255, green=255, blue=255)
2 0 542 505 RGB(red=198, green=111, blue=127)
2 1 582 505 RGB(red=14, green=17, blue=22)
2 2 622 505 RGB(red=14, green=17, blue=22)
2 3 662 505 RGB(red=255, green=255, blue=255)
2 4 702 505 RGB(red=255, green=255, blue=255)
2 5 742 505 RGB(red=255, green=255, blue=255)
2 6 782 505 RGB(red=2

In [3]:
def get_gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [4]:
def get_gem_color(r, c, im):
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [5]:
def can_move_left(r, c, im):
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left_c = c - 1
    left_color = get_gem_color(r, left_c, im)
    current_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, left_c):
            return current_color
        if (row, col) == (r, c):
            return left_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + dr * direction
                nc = col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and color_after_swap(nr, nc) == color):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, left_c) or makes_match(r, c)

In [6]:
def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, 1, 0, im)

In [7]:
im = pyg.screenshot()
print(can_move_left(5, 4, im))

False


In [8]:
print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))

(542, 425)
(582, 425)
(542, 465)


In [9]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, i, im))

(56, 56, 58, 255)
(28, 28, 29, 255)
(28, 28, 29, 255)
(28, 28, 29, 255)
(28, 28, 29, 255)
(28, 28, 29, 255)
(28, 28, 29, 255)
(28, 28, 29, 255)


In [10]:
import time

start_time = time.time()
im = pyg.screenshot()

while True:
    if time.time() - start_time > 60:
        break

    moved = False  # 標記本輪掃描是否有成功移動寶石

    for r in range(8):
        for c in range(8):
            if time.time() - start_time > 60:
                break

            directions = (
                (can_move_left, 0, -1),
                (can_move_right, 0, 1),
                (can_move_up, -1, 0),
                (can_move_down, 1, 0),
            )

            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.2, button='left')  # 修正 1：明確指定 button='left' 解決 macOS 報錯

                    # 延長等待時間，確保消除和掉落動畫已結束
                    pyg.sleep(0.7)

                    im = pyg.screenshot()
                    moved = True
                    break  # 跳出方向迴圈

            if moved:
                break  # 跳出 c 迴圈

        if moved:
            continue  # 修正 2：跳出 r 迴圈後直接重頭執行 while 迴圈（從 0,0 重新掃描新版面）

    # 防呆機制：如果整盤 64 格都沒找到任何可消除的組合，更新截圖並稍微等待，防止死迴圈卡死 CPU
    if not moved:
        pyg.sleep(0.3)
        im = pyg.screenshot()